data preparation

## 3. Data generation for the specific domain
In this module the data samples generated (and labeled). The initial data was gathered, as mentioned, from open web sources, and cleaned. Then it was augmented, using back-translation and word replacement. Overall the number of examples was quadrupled, yielding satisfactory results.


In [1]:
from pathlib import Path
import re
import pandas as pd

### 3.1 cleaning and preparing the data
The data was cleaned from artifacts using regex. The word problems were identified by numbering - these numbers were erased in order to avoid counting the numbers as part of the problem.

In [2]:
def clean_text_and_mark(in_path):
    in_path = Path(in_path)
    with in_path.open("r", encoding="utf-8") as f:
        text = f.read()
    text = re.sub(r'\(\s*\d+\s*\)', '\n&&& ', text)

# replace numbers with a dot before them like .1 or . 123 -> &&&
    text = re.sub(r'\.\s*\d+', '\n&&& ', text)

    out_path = in_path.with_name(f"cleaned_{in_path.name}")
    out_path.write_text(text, encoding="utf-8")

    print(f"Processed file written to {out_path}")
    return text

In [3]:
def load_df(sample):
    text = clean_text_and_mark(sample[0])
    label = sample[1]
    segments = []
    for seg in text.split('&&&'):
        seg = seg.strip()
        if not seg:
            continue
        seg = re.sub(r'\s+', ' ', seg).strip()
        if seg:
            segments.append(seg)

    df = pd.DataFrame({'question': segments, 'label': [label] * len(segments)})
    return df

def load_questions(samples):
    frames = []
    for sample in samples:
        df = load_df(sample)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


In [4]:
samples = [('data/word_problems_8th.txt', '8th'), ('data/word_problems_9th.txt', '9th')]
df = load_questions(samples)

Processed file written to data/cleaned_word_problems_8th.txt
Processed file written to data/cleaned_word_problems_9th.txt


In [5]:
df.shape

(271, 2)

In [15]:
# save original data in csv file
copy = df.copy()
copy['label'] = copy['label'].astype(str).str.strip().replace({'8th': 0, '9th': 1}).astype(int)
copy.to_csv('output/word_problems_no_augmentation.csv', index=False, encoding='utf-8-sig')
print(f"Wrote {len(copy)} rows to output/word_problems_no_augmentation.csv")

Wrote 271 rows to output/word_problems_no_augmentation.csv


/var/folders/vb/0cfv67fd481b5zf98tps8h3h0000gn/T/ipykernel_93687/739968903.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  copy['label'] = copy['label'].astype(str).str.strip().replace({'8th': 0, '9th': 1}).astype(int)


### 3.2 Back translation augmention
In this augmentation method, the text is translated to a different language and then back translated to the original one. The module uses google translation api. For better results a third translation stage was added, so the translation is Hebrew -> English -> Arabic -> Hebrew. This stage takes a while (because of the web api) so i performed it before other augmentation stages.

In [6]:
from deep_translator import GoogleTranslator

def back_translate_api(text):
    """
    Performs Back-Translation using Google Translate API (via deep_translator).
    No local models or heavy dependencies required.
    """
    try:
        # 1. Hebrew -> English
        translated_en = GoogleTranslator(source='iw', target='en').translate(text)
        # 2. English -> Arabic
        translated_mid = GoogleTranslator(source='en', target='ar').translate(translated_en)

        # 3. Arabic -> Hebrew
        translated_he = GoogleTranslator(source='ar', target='iw').translate(translated_mid)

        return translated_he
    except Exception as e:
        print(f"Error during translation: {e}")
        return text

# --- Main Execution ---

problem = "דני קנה 5 תפוחים ושילם עליהם 20 שקלים בסך הכל."

print(f"Original: {problem}")

# Note: Since this uses a web API, it might take a second per request.
new_version = back_translate_api(problem)

print(f"Paraphrased: {new_version}")


Original: דני קנה 5 תפוחים ושילם עליהם 20 שקלים בסך הכל.
Paraphrased: דני קנה 5 תפוחים ושילם עבורם 20 שקלים.


translate

In [7]:
# Interleave an augmented row after each existing row in `df` using the existing `augmenter`.
def back_translate(df):
    orig = df.reset_index(drop=True)

    def _safe_augment(text):
        if pd.isna(text):
            return text
        try:
            return back_translate_api(text)
        except Exception as e:
            print(f"Augmentation failed: {e}")
            return text

    aug = orig.copy()
    aug['question'] = orig['question'].apply(_safe_augment)

    # put originals at even indices and augmented at odd indices, then recombine
    orig.index = orig.index * 2
    aug.index = aug.index * 2 + 1

    df = pd.concat([orig, aug]).sort_index().reset_index(drop=True)
    print(f"Done. New df shape: {df.shape}")
    return df

In [8]:
augmented_df = back_translate(df) # takes a while - a second per question

Error during translation: None --> text must be a valid text with maximum 5000 character,otherwise it cannot be translated
Error during translation: None --> text must be a valid text with maximum 5000 character,otherwise it cannot be translated
Done. New df shape: (542, 2)


### 3.3 Word replacement augmentation
In this stage every sample is proccessed with the stanza nlp library, and some words and names are replaced.

In [9]:
import stanza
import random
from faker import Faker

stanza.download('he') 

nlp = stanza.Pipeline(lang='he', processors='tokenize,ner')
fake = Faker('he_IL')


/Users/erannovak/miniconda3/envs/openUNLP/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-11 08:47:21 INFO: Downloaded file to /Users/erannovak/stanza_resources/resources.json
2026-02-11 08:47:21 INFO: Downloading default packages for language: he (Hebrew) ...
2026-02-11 08:47:22 INFO: File exists: /Users/erannovak/stanza_resources/he/default.zip
2026-02-11 08:47:24 INFO: Finished downloading models and saved to /Users/erannovak/stanza_resources
2026-02-11 08:47:24 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-02-11 08:47:24 INFO: Downloaded file to /Users/erannovak/stanza_resources/resources.json
2026-02-11 08:47:24 WARNING: Language he package

replacing words and names.

In [10]:

class HebrewMathAugmenter:
    def __init__(self):
        self.synonyms = {
            'קנה': ['רכש', 'השיג'],
            'מכר': ['שיווק'],
            'קיבל': ['קיבל לידיו', 'אסף'],
            'סה"כ': ['בסך הכל', 'ביחד'],
            'כמה': ['מה כמות', 'מה מספר'],
            'נותרו': ['נשארו'],
            'ידוע' : ['נתון כי']
        }

    def _replace_names(self, text):
        doc = nlp(text)
        name_map = {}
        
        for ent in doc.ents:
            if ent.type in ['PER', 'S-PER']:
                original_name = ent.text
                if original_name not in name_map:
                    new_name = fake.first_name() 
                    name_map[original_name] = new_name
        
        augmented_text = text
        for old, new in name_map.items():
            augmented_text = re.sub(rf'\b{re.escape(old)}\b', new, augmented_text)
            
        return augmented_text

    def _replace_synonyms(self, text):
        words = text.split()
        new_words = []
        for word in words:
            clean_word = re.sub(r'[^\w]', '', word)
            
            if clean_word in self.synonyms and random.random() > 0.5:
                replacement = random.choice(self.synonyms[clean_word])
                word = word.replace(clean_word, replacement)
            new_words.append(word)
        return ' '.join(new_words)

    # def _shuffle_sentences(self, text):
    #     """שינוי סדר המשפטים (בזהירות!)"""
    #     sentences = [s.strip() for s in text.split('.') if s.strip()]
        
    #     if len(sentences) > 1:
    #         question = sentences[-1]
    #         body = sentences[:-1]
    #         random.shuffle(body)
    #         body.append(question)
    #         return '. '.join(body) + '.'
        
    #     return text

    def augment(self, text):
        text = self._replace_names(text)
        text = self._replace_synonyms(text)
        return text

augmenter = HebrewMathAugmenter()

In [11]:
# Interleave an augmented row after each existing row in `df`.
# Uses the existing `augmenter` object (HebrewMathAugmenter) and `pd` from earlier cells.

orig = augmented_df.reset_index(drop=True)

aug_questions = []
for q in orig['question']:
    if pd.isna(q):
        aug_questions.append(q)  # preserve NaN
    else:
        try:
            aug_questions.append(augmenter.augment(q))
        except Exception as e:
            # if augmentation fails, keep the original question
            print(f"Augmentation failed for question: {q!r}: {e}")
            aug_questions.append(q)

aug = orig.copy()
aug['question'] = aug_questions

# place originals at even positions, augmented at odd positions, then sort
orig.index = orig.index * 2
aug.index = aug.index * 2 + 1

final_df = pd.concat([orig, aug]).sort_index().reset_index(drop=True)

print(f"Done. New df shape: {final_df.shape}")

Done. New df shape: (1084, 2)


Changing labels to 0, 1

In [12]:
# convert textual labels to integers in-place
final_df['label'] = final_df['label'].astype(str).str.strip().replace({'8th': 0, '9th': 1}).astype(int)

# quick check
print(final_df['label'].value_counts())

label
1    588
0    496
Name: count, dtype: int64


/var/folders/vb/0cfv67fd481b5zf98tps8h3h0000gn/T/ipykernel_93687/2609329567.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_df['label'] = final_df['label'].astype(str).str.strip().replace({'8th': 0, '9th': 1}).astype(int)


In [ ]:
final_df.to_csv('output/word_problems_table.csv', index=False, encoding='utf-8-sig')
print(f"Wrote {len(final_df)} rows to output/word_problems_table.csv")

Wrote 1084 rows to output/word_problems_table.csv


Total of 1084 samples of word problems.